## Assessment 2

Use dataset from Assessment 1 to train a machine learning model on historical records then appluy it to a simulated real-time stream. This assessment supports unit learning outcomes 3,4 and 5. 

### Before Part A
Data is split into training dataset, 70% of records, to train and evaluate the ML model and remaining 30% will be used for streaming dataset and will be replayed through Kfka in Part B.

In [272]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext

df = spark.read.csv("ncr_ride_bookings.csv",header=True)

# Replace string 'null' with true None/null globally
df = df.replace("null", None)



In [273]:
train_df1, stream_df = df.randomSplit([0.7,0.3],seed = 42)

#to ensure a 70/30 split of data
print(train_df1.count())
print(train_df1.count()/150000)
print(stream_df.count())
print(stream_df.count()/150000)

#repeat of assessment one to ensure data type is correct
from pyspark.sql.functions import col,concat_ws
from pyspark.sql.types import DateType

float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Rating","Booking Value"]
train_df1 = train_df1.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])

#remove all entries that do no have a driver rating number, which is the number the customer gives to the driver
#this is done after the initial split to reflect what real-time data would be, as the driver rating is optional for customers
train_df = train_df1.dropna(subset=["Driver Ratings"])
print(train_df.count())

105209
0.7013933333333333
44791
0.2986066666666667
65201


### Part A - The ML Pipeline

#### 1. The ML task selection

The ideal machine learning algorithm for the original business case of trying to predict the driver rating of their ride based on the record features would be regression. This is becuase the ratings are numerical in value, continuous after some data cleaning, and do have meaningful variance. The dataset included will only included records that do have a driver rating, which is much lower than the original dataset but still large enough to create a model with meaningful result. 

#### 2. Feature Enginerring

For this dataset, there are four general types of data that is included - categorical, binary, numerical and free text. Free text is seperated as it required further work to categorise this data, such as feature extraction and feature transformation. Binary is also removed, as it can easily be converted to full numerical with NULL coverted to 0. Booking ID and Customer ID are removed, as they are unique and identifying features that would not provide any predictable value. 

In [274]:
train_df.show(5)

+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|    Time|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-------------

In [275]:
#change the date and time into categorical values that would be impactful in prediction, specifically month, day and hour
#change hour, dayofweek and month into strings

from pyspark.sql.functions import hour, dayofweek, month

train_df = (train_df.withColumn("Hour", hour("Time").cast("string")).withColumn("DayOfWeek", dayofweek("Date").cast("string"))
.withColumn("Month", month("Date").cast("string")))
train_df.show(5)

+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Date|    Time|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+---------------

In [276]:
train_df = train_df.drop("Date", "Time")
train_df.show(5)

+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------

The date and time were converted into hour, day of the week and month as this will be more relevant to the final model. Even though they are numerical values representing specific features, they will be treated at categories as they represent a specific time of day or year rather than a numerical number. 

In [277]:
#seperate numerically relevant columns from category columns as well as binary columns and free text
categoryInputCols = ["Booking Status", "Vehicle Type", "Pickup Location", "Drop Location","Payment Method", "Hour", "DayOfWeek", "Month"]
binaryInputCols = ["Cancelled Rides by Customer","Cancelled Rides by Driver", "Incomplete Rides"]
freetextInputCols = ["Reason for cancelling by Customer","Driver Cancellation Reason","Incomplete Rides Reason"]
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Ratings"]
numericOutputCol = "Driver Ratings"

numericalCols = [numericOutputCol]+numericInputCols

In [278]:
train_df.select(binaryInputCols).printSchema()

root
 |-- Cancelled Rides by Customer: string (nullable = true)
 |-- Cancelled Rides by Driver: string (nullable = true)
 |-- Incomplete Rides: string (nullable = true)



In [279]:
#covert binary input cols into full numerical with Null = 0.0. since column is string, need to state that rest of values are numerical

from pyspark.sql.functions import when

for c in binaryInputCols:
    train_df = train_df.withColumn(c,when(col(c).isNull(), 0.0).otherwise(col(c).cast("double")))
train_df.show()

+----------------+--------------+----------------+------------+-------------------+--------------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Booking ID|Booking Status|     Customer ID|Vehicle Type|    Pickup Location|       Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------------+--------------+----------------+------------+-------------------+--------------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+---------------

#### 2a. Investigation into Payment method

As part of feature extraction, I needed to investiagte paymend method and impact on the specific part of the dataset I was looking at. Due to that, I went back and investigated the impact of removing all null Driver Ratings from above and reassessed what columns I needed within my model. After the below investigation, the concluding assumption is that only rides that are complete allow the customer to give the driver a rating. A rating cannot be given if the booking is incomplete or cancelled, however if it is incomplete a payment is still taken and recorded. Based on this investigation, it was determined that Booking Status and any information relating to cancellation or incomplete would have no influence on the predictive model to determine Driver Ratings. Therefore these will be removed from the features used in model.

In [280]:
#Review null values in category columns to see if data is missing or important

#only looking at the non-altered categories 
null_categoryInputCols = ["Booking Status", "Vehicle Type", "Pickup Location", "Drop Location","Payment Method"]

from pyspark.sql.functions import col, sum
print("dataset before the removing of null driver ratings")
train_df1.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("dataset after the removing of null driver ratings")
train_df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("streaming dataset with null")
stream_df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("total dataset with null")
df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()

dataset before the removing of null driver ratings
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location|Drop Location|Payment Method|
+--------------+------------+---------------+-------------+--------------+
|             0|           0|              0|            0|         33638|
+--------------+------------+---------------+-------------+--------------+

dataset after the removing of null driver ratings
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location|Drop Location|Payment Method|
+--------------+------------+---------------+-------------+--------------+
|             0|           0|              0|            0|             0|
+--------------+------------+---------------+-------------+--------------+

streaming dataset with null
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location

In [281]:
from pyspark.sql.functions import col, count

df.groupBy("Booking Status").agg(
    sum(col("Driver Ratings").isNull().cast("int")).alias("Missing Driver Rating"),
    sum(col("Payment Method").isNull().cast("int")).alias("Missing Payment Method"),
    count("*").alias("Total")
).show()

+--------------------+---------------------+----------------------+-----+
|      Booking Status|Missing Driver Rating|Missing Payment Method|Total|
+--------------------+---------------------+----------------------+-----+
|           Completed|                    0|                     0|93000|
|     No Driver Found|                10500|                 10500|10500|
| Cancelled by Driver|                27000|                 27000|27000|
|Cancelled by Cust...|                10500|                 10500|10500|
|          Incomplete|                 9000|                     0| 9000|
+--------------------+---------------------+----------------------+-----+



#### 2b. Vector Assesmbler

In [282]:
#final set of columns that will be used in vector assessmbler and model generation
#seperate numerically relevant columns from category columns as well as binary columns and free text
categoryInputCols = ["Vehicle Type", "Pickup Location", "Drop Location","Payment Method","Hour","DayOfWeek","Month"]
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Rating"]
numericOutputCol = "Driver Ratings"

numericalCols = [numericOutputCol]+numericInputCols

In [283]:
from pyspark.ml.feature import StringIndexer

# Define output column names
outputCols = [f'{x}_index' for x in categoryInputCols]

print(categoryInputCols)
print(outputCols)

# Create index values for categorical input features
inputIndexer = StringIndexer(inputCols=categoryInputCols,outputCols=outputCols,handleInvalid="keep")

# Call the fit and transform() method to get the encoded results 
train_df_indexed = inputIndexer.fit(train_df).transform(train_df)
# Display the output, only the output columns
train_df_indexed.select(outputCols).show(10)

['Vehicle Type', 'Pickup Location', 'Drop Location', 'Payment Method', 'Hour', 'DayOfWeek', 'Month']
['Vehicle Type_index', 'Pickup Location_index', 'Drop Location_index', 'Payment Method_index', 'Hour_index', 'DayOfWeek_index', 'Month_index']
+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+
|Vehicle Type_index|Pickup Location_index|Drop Location_index|Payment Method_index|Hour_index|DayOfWeek_index|Month_index|
+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+
|               1.0|                104.0|              118.0|                 4.0|      18.0|            5.0|        0.0|
|               2.0|                 16.0|              120.0|                 0.0|      12.0|            5.0|        0.0|
|               0.0|                 95.0|               23.0|                 4.0|       8.0|            5.0|        0.0|
|               3.

With the first step in feature extraction, the StringIndexer is converting categories into numerical values. It uses the column names identified above and assigns each type in that column a numerical number. For example, with Vechile Type there are approximately 7 different types and therefore the index assigned is 0-6. However, for location there are a lot more types available and therefore the numerical range is a lot higher.  

In [284]:
from pyspark.ml.feature import OneHotEncoder

# Input columns are the indexed categorical input columns
inputCols_OHE = outputCols

# Define output columns for the one-hot encoded vectors
outputCols_OHE = [f'{x}_vec' for x in categoryInputCols]

#Define OneHotEncoder with the appropriate columns
encoder = OneHotEncoder(inputCols=inputCols_OHE,outputCols=outputCols_OHE)

# Call fit and transform to get the encoded results
model = encoder.fit(train_df_indexed)
train_df_encoded = model.transform(train_df_indexed)
# Display the output columns
train_df_encoded.select(outputCols_OHE).show(5)

+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+
|Vehicle Type_vec|Pickup Location_vec|Drop Location_vec|Payment Method_vec|       Hour_vec|DayOfWeek_vec|     Month_vec|
+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+
|   (7,[1],[1.0])|  (176,[104],[1.0])|(176,[118],[1.0])|     (5,[4],[1.0])|(24,[18],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[2],[1.0])|   (176,[16],[1.0])|(176,[120],[1.0])|     (5,[0],[1.0])|(24,[12],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[0],[1.0])|   (176,[95],[1.0])| (176,[23],[1.0])|     (5,[4],[1.0])| (24,[8],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[3],[1.0])|  (176,[169],[1.0])| (176,[67],[1.0])|     (5,[2],[1.0])| (24,[3],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[0],[1.0])|   (176,[96],[1.0])| (176,[70],[1.0])|     (5,[1],[1.0])| (24,[6],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
+----------------+--------------

The second part requires coverting the numerical data of each of the features into a sparse vector that stores non-zero values and their indices. For each vector the first value is the number of types available in that specific feature, so for Vechicle Type the first value for all vectors in 7, then the indices of where the value is within the all values, and finally the value itself. Since each vector is a single, the value for all entries will be 1.0. 

In [299]:
from pyspark.ml.feature import VectorAssembler

# inputCols are all the encoded columns from OHE plus numerical columns
assemblerInputs = outputCols_OHE + numericInputCols

# Combine all input features into a single feature vector
assembler = VectorAssembler(inputCols=assemblerInputs,outputCol="features")

# Transform the encoded dataframe
train_df_final = assembler.transform(train_df_encoded)

# Display features and numerical output
train_df_final.select("features", numericOutputCol).show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|features                                                                                                                                                            |Driver Ratings|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|(412,[1,111,301,363,382,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.800000190734863,44.0,261.0,25.030000686645508,4.5])                             |4.2           |
|(412,[2,23,303,359,376,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,14.699999809265137,23.100000381469727,344.0,16.760000228881836,4.900000095367432]) |3.8           |
|(412,[0,102,206,363,372,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,11.5,16

The final dataset is created using the vector assesmbler, which calls the sparse vector created above and combines all the features into a single sparse vector for all the features. The sparse vector still have 3 sections but now there are 412 values for each data entry. The final part of the sparse vector shows that the first part are categorical values, since they have 1.0 as the entry, but then goes to numerical values. Finally each data entry is compared to the output column to be used in the model, Drivers Rating.

#### 3/4. Model Comparision and Evaluation

Several models are created and compared using RMSE, R2 and MAE suitable for a regression model. RMSE or root mean square error is the square of the differences between predicted and actual value. For an ideal model, this value should be as close to zero as possible. R-squared is the variance of the predicted value. Close to 1 R2 is the better the model to fitting the dataset. Finally mean absolute error or MAE is the absolute difference between predicted and true value. It is simialr to RMSE but this is a more direct way of interpreting the accuracy of the model and ideal value would be close to 0 as possible. Since the model is trying to predict Drivers Rating, which is between 1-5, this would give an immediate insight into the model and how well it fits to training data. 

In [301]:
#train_df is further split into 70/30 to allow models to be trained. This training dataset will be used for all models. 
train, test = train_df_final.randomSplit([0.7, 0.3], seed = 2020)
print("Training Dataset Count: " + str(train.count()))
print("Test Dataset Count: " + str(test.count()))

train.show(5)


Training Dataset Count: 45555
Test Dataset Count: 19646
+----------------+--------------+----------------+-------------+---------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+--------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type|Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Cu

In [320]:
#creating a function that i can use for all evaluations of my regression models
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate_regression(predictions, labelCol):

    rmse_evaluator = RegressionEvaluator(labelCol=labelCol,predictionCol="prediction",metricName="rmse")
    r2_evaluator = RegressionEvaluator(labelCol=labelCol,predictionCol="prediction",metricName="r2")
    mae_evaluator = RegressionEvaluator(labelCol=labelCol, predictionCol = "prediction",metricName= "mae")
    
    rmse = rmse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")
    print(f"MAE:   {mae:.3f}")
    
    return

In [321]:
from pyspark.ml.regression import LinearRegression

# Create an initial model using the train set, using basic linear regression with no additional settings
lr = LinearRegression(featuresCol = "features", labelCol=numericOutputCol)
lrModel = lr.fit(train)

lrPredictions = lrModel.transform(test)
lrPredictions.select(numericOutputCol,'prediction').show()

26/10/03 11:53:10 WARN Instrumentation: [c98b4636] regParam is zero, which might cause numerical instability and overfitting.
26/10/03 11:53:11 WARN Instrumentation: [c98b4636] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


+--------------+------------------+
|Driver Ratings|        prediction|
+--------------+------------------+
|           4.4| 4.149727015335586|
|           4.6| 4.290834459055676|
|           4.3| 4.265141597746973|
|           4.2| 4.216547416156797|
|           4.1| 4.212980762453868|
|           4.2| 4.248500940760164|
|           4.6| 4.229237735088287|
|           4.7| 4.244978009208971|
|           4.5| 4.258383070388347|
|           3.2| 4.201292830464315|
|           4.1| 4.183222754261617|
|           4.4| 4.249449889676868|
|           4.9| 4.189052588006494|
|           4.7| 4.199448337906713|
|           4.5| 4.323200864801347|
|           4.2| 4.276633147727598|
|           5.0| 4.233456181080404|
|           3.1| 4.243305908534895|
|           4.1|4.2963254979189935|
|           3.8|  4.20956336540194|
+--------------+------------------+
only showing top 20 rows


In [322]:
evaluate_regression(lrPredictions_1,numericOutputCol)

RMSE: 0.439
R²:   -0.009
MAE:   0.333


In [331]:
#introducing regularisation to control the fit of the model
lr_l2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol,regParam=0.1,elasticNetParam=0.0)
lrl2_model = lr_l2.fit(train)
lrl2_Predictions = lrl2_model.transform(test)
evaluate_regression(lrl2_Predictions,numericOutputCol)


RMSE: 0.439
R²:   -0.006
MAE:   0.332


In [326]:
print(len(train.select("features").first()["features"]))

412


Based on the large number of features within the df_final dataset, mainly due to the large number of locations possible in the pickup and drop-off location, a seperate dataset will be created without those columns to see if this could create a better suited model.

In [341]:
categoryInputCols_nolocation = ["Vehicle Type","Payment Method","Hour","DayOfWeek","Month"]

# Define output column names
outputCols = [f'{x}_index' for x in categoryInputCols_nolocation]

# Create index values for categorical input features
inputIndexer1 = StringIndexer(inputCols=categoryInputCols_nolocation,outputCols=outputCols,handleInvalid="keep") 
train_df_indexed_noloc = inputIndexer1.fit(train_df).transform(train_df)

# Input columns are the indexed categorical input columns
inputCols_OHE = outputCols
outputCols_OHE = [f'{x}_vec' for x in categoryInputCols_nolocation]

#Define OneHotEncoder with the appropriate columns
encoder1 = OneHotEncoder(inputCols=inputCols_OHE,outputCols=outputCols_OHE)
model1 = encoder1.fit(train_df_indexed_noloc)
train_df_encoded1 = model1.transform(train_df_indexed_noloc)

# inputCols are all the encoded columns from OHE plus numerical columns
assemblerInputs1 = outputCols_OHE + numericInputCols
assembler1 = VectorAssembler(inputCols=assemblerInputs1,outputCol="features")
train_df_noloc= assembler1.transform(train_df_encoded1)

#number of features within the new dataset
print(len(train_df_noloc.select("features").first()["features"]))

train_noloc, test_noloc = train_df_noloc.randomSplit([0.7, 0.3], seed = 2020)
print("No location training Dataset Count: " + str(train.count()))
print("No location test Dataset Count: " + str(test.count()))

60
No location training Dataset Count: 45555
No location test Dataset Count: 19646


In [342]:
#repeat of linear regression with no location dataset
lr2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol)
lr2_model = lr2.fit(train_noloc)
lr2_Predictions = lr2_model.transform(test_noloc)
evaluate_regression(lr2_Predictions,numericOutputCol)

26/10/03 12:11:45 WARN Instrumentation: [16d169d0] regParam is zero, which might cause numerical instability and overfitting.
26/10/03 12:11:45 WARN Instrumentation: [16d169d0] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


RMSE: 0.438
R²:   -0.001
MAE:   0.331


In [343]:
#repeat with regularisation parameter
lr2_l2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol,regParam=0.0001,elasticNetParam=0.0)
lr2_l2_model = lr2_l2.fit(train_noloc)
lr2_l2_Predictions = lr2_l2_model.transform(test_noloc)
evaluate_regression(lr2_l2_Predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.001
MAE:   0.331


In [344]:
#trying a decision tree model to see if that improves the fitting
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,seed=42)

# Train model with no location data
dt_model = dt.fit(train_noloc)
dt_predictions = dt_model.transform(test_noloc)

dt_results = evaluate_regression(dt_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.004
MAE:   0.332


In [351]:
dt1 = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,maxDepth=5,minInstancesPerNode=10,seed=42)

# Train model with no location data
dt1_model = dt1.fit(train_noloc)
dt1_predictions = dt1_model.transform(test_noloc)

dt1_results = evaluate_regression(dt1_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.004
MAE:   0.332


In [352]:
dt2 = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,maxDepth=15,minInstancesPerNode=5,seed=42)

# Train model with no location data
dt2_model = dt2.fit(train_noloc)
dt2_predictions = dt2_model.transform(test_noloc)

dt2_results = evaluate_regression(dt2_predictions,numericOutputCol)

RMSE: 0.465
R²:   -0.132
MAE:   0.357
